# Import libraries

In [1]:
from pyspark.sql.functions import expr
from pyspark.sql.types import IntegerType
from pyspark.sql.functions import upper, col, when

# Read customer profiles data using spark dataframe reader

In [2]:
customers = spark.read.table("churn_bronze.crm.customers")
display_cols = ["customerid", "gender", "monthlycharges", "totalcharges"]
display(customers.select(display_cols))

# Read customer reviews data

In [3]:
customer_reviews = spark.read.table("churn_bronze.crm.customer_reviews")
display(customer_reviews)

# Join Customer profiles and reviews dataframe

In [4]:
customer_df = customers.join(customer_reviews, "customerid") 
customer_df.printSchema()

root
 |-- customerid: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- seniorcitizen: integer (nullable = true)
 |-- partner: string (nullable = true)
 |-- dependents: string (nullable = true)
 |-- tenure: integer (nullable = true)
 |-- phoneservice: string (nullable = true)
 |-- multiplelines: string (nullable = true)
 |-- internetservice: string (nullable = true)
 |-- onlinesecurity: string (nullable = true)
 |-- onlinebackup: string (nullable = true)
 |-- deviceprotection: string (nullable = true)
 |-- techsupport: string (nullable = true)
 |-- streamingtv: string (nullable = true)
 |-- streamingmovies: string (nullable = true)
 |-- contract: string (nullable = true)
 |-- paperlessbilling: string (nullable = true)
 |-- paymentmethod: string (nullable = true)
 |-- monthlycharges: double (nullable = true)
 |-- totalcharges: double (nullable = true)
 |-- review: string (nullable = true)
 |-- review_date: date (nullable = true)



# Select the relevant attributes

In [5]:
from pyspark.sql.functions import col

# Select the specified columns
selected_columns = [
    "customerid", 
    "gender", 
    "seniorcitizen", 
    "partner", 
    "dependents", 
    "tenure", 
    "phoneservice", 
    "multiplelines", 
    "internetservice", 
    "onlinesecurity", 
    "onlinebackup", 
    "deviceprotection", 
    "techsupport", 
    "streamingtv", 
    "streamingmovies", 
    "contract", 
    "paperlessbilling", 
    "paymentmethod", 
    "monthlycharges", 
    "totalcharges", 
    "review"
]
customers_selected = customer_df.select(*[col(c) for c in selected_columns])

# Drop rows with blank columns

In [6]:
from pyspark.sql import functions as F

# Drop rows with NA values in filtered_df and name the result filtered_customers
filtered_customers = customers_selected.dropna()
filtered_customers = filtered_customers.limit(100)

# Call GenAI model for sentiment analysis of customer reviews

In [7]:
filtered_customers = filtered_customers.withColumn('sentimentScore_str', expr("query_model('default.oci_ai_models.meta.llama-3.2-90b-vision-instruct', concat('What is the sentiment of the review text on a scale of 1 to 5, please give the output as an integer only', review))"))

In [8]:
filtered_customers = filtered_customers.withColumn("sentimentScore", filtered_customers["sentimentScore_str"].cast(IntegerType()))
filtered_customers = filtered_customers.drop("sentimentScore_str")

# Create crm schema

In [9]:
spark.sql("CREATE SCHEMA IF NOT EXISTS churn_silver.crm").show()

++
||
++
++



# Save cleansed and transformed data to silver layer

In [10]:
table_name = "churn_silver.crm.customers"
filtered_customers = filtered_customers.dropna()
filtered_customers.write.mode("overwrite").format("delta").saveAsTable(table_name)

In [11]:
spark.table(table_name).count()

100